In [22]:
import pandas as pd
import numpy as np

In [23]:
data_healthy_one = pd.read_csv(r"..\data\raw\healthy\span1.CSV")
data_healthy_two = pd.read_csv(r"..\data\raw\healthy\span2.CSV")
data_healthy_three = pd.read_csv(r"..\data\raw\healthy\span3.CSV")
data_defective_one = pd.read_csv(r"..\data\raw\defective\span1.CSV")
data_defective_two = pd.read_csv(r"..\data\raw\defective\span2.CSV")
data_defective_three = pd.read_csv(r"..\data\raw\defective\span3.CSV")

In [24]:
def add_labels(df, condition):
    """Add a 'condition' column (healthy/defective) and a 'recording' column.

    Recordings are named RH1, RH2, ... in healthy files and RD1, RD2, ... in
    defective files.

    pandas used the FIRST header row as column names. Every later header row
    (the start of a new recording) was read in as an ordinary row, with the
    text "Time(ms)" sitting in the Time(ms) column. We use those rows to count
    recordings, then remove them.
    """
    # Safety: if this DataFrame was already labelled (e.g. you re-ran this cell),
    # the header rows are gone, so labelling again would mark everything as recording 1.
    if "recording" in df.columns:
        return df

    df = df.copy()

    # True on each repeated header row, False on normal data rows
    is_header = df["Time(ms)"] == "Time(ms)"

    # Running count of header rows seen so far: 0 in the first recording,
    # 1 in the second, and so on. Add 1 so numbering starts at 1.
    recording_number = is_header.cumsum() + 1

    # "RH" for healthy files, "RD" for defective files
    prefix = "RH" if condition == "healthy" else "RD"

    df["condition"] = condition
    df["recording"] = prefix + recording_number.astype(str)

    # Drop the repeated header rows
    df = df[~is_header].reset_index(drop=True)

    # Because of those header rows, pandas stored the numbers as text.
    # Now that they're gone, convert the columns back to numbers.
    numeric_cols = ["Time(ms)", "Ax(g)", "Ay(g)", "Az(g)", "Accel_Magnitude(g)"]
    df[numeric_cols] = df[numeric_cols].astype(float)

    return df


data_healthy_one = add_labels(data_healthy_one, "healthy")
data_healthy_two = add_labels(data_healthy_two, "healthy")
data_healthy_three = add_labels(data_healthy_three, "healthy")
data_defective_one = add_labels(data_defective_one, "defective")
data_defective_two = add_labels(data_defective_two, "defective")
data_defective_three = add_labels(data_defective_three, "defective")

In [25]:
# Quick check: how many recordings did we find in each file?
# Expected: healthy 12, 10, 12 (= 34) and defective 8, 5, 5 (= 18)
all_dfs = {
    "healthy_one": data_healthy_one,
    "healthy_two": data_healthy_two,
    "healthy_three": data_healthy_three,
    "defective_one": data_defective_one,
    "defective_two": data_defective_two,
    "defective_three": data_defective_three,
}
for name, df in all_dfs.items():
    print(f"{name}: {df['recording'].nunique()} recordings, {len(df)} rows, condition = {df['condition'].unique()}")

data_healthy_one.head()

healthy_one: 12 recordings, 1756 rows, condition = <StringArray>
['healthy']
Length: 1, dtype: str
healthy_two: 10 recordings, 1922 rows, condition = <StringArray>
['healthy']
Length: 1, dtype: str
healthy_three: 12 recordings, 2326 rows, condition = <StringArray>
['healthy']
Length: 1, dtype: str
defective_one: 8 recordings, 1222 rows, condition = <StringArray>
['defective']
Length: 1, dtype: str
defective_two: 5 recordings, 843 rows, condition = <StringArray>
['defective']
Length: 1, dtype: str
defective_three: 5 recordings, 828 rows, condition = <StringArray>
['defective']
Length: 1, dtype: str


,Time(ms),Ax(g),Ay(g),Az(g),Accel_Magnitude(g),condition,recording
0,1.0,-0.0527,1.0889,1.0095,1.4858,healthy,RH1
1,71.0,-0.0159,1.1675,1.0288,1.5562,healthy,RH1
2,140.0,-0.0527,1.1567,0.9524,1.4993,healthy,RH1
3,210.0,-0.0232,1.1274,0.9692,1.4870,healthy,RH1
4,281.0,-0.0105,1.1865,0.9907,1.5458,healthy,RH1
